# oxDNA: Multi-Trajectory Propeller Twist Optimization

This notebook demonstrates multi-trajectory DiffTRe optimization of the propeller twist observable using multiple parallel oxDNA simulations coordinated via Ray.

## What is mythos?

**[Mythos](https://github.com/mythos-bio/mythos)** is a differentiable molecular simulation framework for parameterizing coarse-grained force fields. It provides automatic differentiation through simulation workflows — enabling gradient-based optimization of force field parameters to match experimental or all-atom reference data.

This notebook shows how to run multiple oxDNA simulations in parallel and aggregate their gradients to optimize propeller twist parameters. The `RayOptimizer` coordinates multiple simulators and objectives through Ray.

## Imports

In [ ]:
import logging
import operator
import typing
from pathlib import Path

import jax
import jax.numpy as jnp
import jax_md
import mythos.energy as jdna_energy
import mythos.energy.dna1 as dna1_energy
import mythos.input.topology as jdna_top
import mythos.observables as jd_obs
import mythos.optimization.objective as jdna_objective
import mythos.optimization.optimization as jdna_optimization
import mythos.utils.types as jdna_types
import optax
import ray
from mythos.simulators import oxdna
from mythos.ui.loggers.console import ConsoleLogger

jax.config.update("jax_enable_x64", True)
logging.basicConfig(level=logging.INFO)
logging.getLogger("jax").setLevel(logging.WARNING)

## Configuration

In [ ]:
N_OPT_STEPS = 20
N_OXDNA_RUNS = 3
LEARNING_RATE = 1e-3
TEMPLATE_DIR = Path("../../data/templates/simple-helix").resolve()
OXDNA_SRC = Path("../../../oxDNA").resolve()

## Initialize Ray

In [ ]:
ray.init(
    log_to_driver=True,
    runtime_env={"env_vars": {"JAX_ENABLE_X64": "True"}},
)

## Build energy function

`create_default_energy_fn` assembles the full oxDNA1 composed energy function from the default parameter sets for the given topology.

In [ ]:
top = jdna_top.from_oxdna_file(TEMPLATE_DIR / "sys.top")

energy_fn = dna1_energy.create_default_energy_fn(
    topology=top,
).with_noopt("ss_stack_weights", "kt")

transform_fn = energy_fn.energy_fns[0].transform_fn
opt_params = energy_fn.opt_params()

## Create parallel simulators

We create multiple `oxDNASimulator` instances using `create_n` — each with a
unique name. The `RayOptimizer` will coordinate running them in parallel via
Ray.

**Note**: By default the oxDNASimulator will copy files from `input_dir` and run
the simulation in a temporary directory. This is particularly useful in a
multi-simulation setting to avoid file conflicts. Due to this, the `source_path`
should be given as an absolute path.

In [ ]:
simulators = oxdna.oxDNASimulator.create_n(
    N_OXDNA_RUNS,
    input_dir=TEMPLATE_DIR,
    energy_fn=energy_fn,
    source_path=OXDNA_SRC,
)

## Define the propeller twist objective

Note that the loss function `prop_twist_loss_fn` is defined in accordance to the
API dictated by `DiffTReObjective`. Note also we provide an aggregate function
for combining gradients across the multiple objectives which is required by the
`RayOptimizer`. In this case, we simply average the gradients across the
trajectories (and in fact we only have one objective, but this pattern
generalizes to multiple objectives each with multiple trajectories).

In [ ]:
prop_twist_fn = jd_obs.propeller.PropellerTwist(
    rigid_body_transform_fn=transform_fn,
    h_bonded_base_pairs=jnp.array([[1, 14], [2, 13], [3, 12], [4, 11], [5, 10], [6, 9]]),
)


def prop_twist_loss_fn(
    traj: jax_md.rigid_body.RigidBody,
    weights: jnp.ndarray,
    energy_model: jdna_energy.base.ComposedEnergyFunction,
    _opt_params,
    _observables,
) -> tuple[float, tuple[str, typing.Any]]:
    obs = prop_twist_fn(traj)
    expected_prop_twist = jnp.dot(weights, obs)
    loss = jnp.sqrt((expected_prop_twist - jd_obs.propeller.TARGETS["oxDNA"]) ** 2)
    return loss, (("prop_twist", expected_prop_twist), {})


def tree_mean(trees: tuple[jdna_types.PyTree]) -> jdna_types.PyTree:
    if len(trees) <= 1:
        return trees[0]
    summed = jax.tree.map(operator.add, *trees)
    return jax.tree.map(lambda x: x / len(trees), summed)


required_obs = [name for sim in simulators for name in sim.exposes()]

propeller_twist_objective = jdna_objective.DiffTReObjective(
    name="prop_twist",
    required_observables=required_obs,
    logging_observables=["loss", "prop_twist", "neff"],
    grad_or_loss_fn=prop_twist_loss_fn,
    energy_fn=energy_fn,
    min_n_eff_factor=0.95,
    max_valid_opt_steps=10,
)

## Run the optimization

The Ray optimizer coordinates running simulators and objectives in parallel
either locally or across a cluster of machines. Each component is run as soon as
its dependencies are ready, and results are aggregated synchronously (on the
driver) once all objectives have become available. The `aggregate_grad_fn` (in
this case `tree_mean`) averages gradients from multiple objectives.

In [ ]:
opt = jdna_optimization.RayOptimizer(
    objectives=[propeller_twist_objective],
    simulators=simulators,
    optimizer=optax.adam(learning_rate=LEARNING_RATE),
    aggregate_grad_fn=tree_mean,
    logger=ConsoleLogger(),
)

output = opt.run(params=opt_params, n_steps=N_OPT_STEPS)
print("\nOptimization complete!")
print(f"Final params: {output.opt_params}")